In [18]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    cross_val_score,
    GroupKFold
)

import optuna
from optuna.samplers import TPESampler

import numpy as np

import os
#os.environ["TQDM_DISABLE"] = "True"
import warnings

In [19]:
def prepare_data(df, categorical_cols, boolean_cols):
    df_new = df.drop(columns = ["timeInterval_ID"])
    X = df_new.drop(columns=['phq8_score', 'participant_name'])
    y = df_new['phq8_score']
    groups = df_new['participant_name']
    cols_to_encode = categorical_cols + boolean_cols
    for col in cols_to_encode:
        X[col] = X[col].cat.codes
        X[col] = X[col].replace(-1, np.nan)
        X[col] = X[col].astype('category')

    return X, y, groups

def get_oh_encoder(all_cat_cols, X):
    categorical_cols_dict = {
        col: sorted(X[col].dropna().unique().tolist())
        for col in all_cat_cols
    }
    categories_list = list(categorical_cols_dict.values())
    oh_encoder = OneHotEncoder(
        categories=categories_list,
        sparse_output=False,
    )
    return oh_encoder

def create_pipeline(X, categorical_cols, boolean_cols, numerical_cols):

    oh_encoder = get_oh_encoder(categorical_cols, X)

    # numeric pipeline
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    # categorical pipeline
    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", oh_encoder)
    ])

    # boolean pipeline
    boolean_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numerical_cols),
            ("cat", categorical_pipeline, categorical_cols),
            ("bool", boolean_pipeline, boolean_cols),
        ],
        verbose_feature_names_out=False
    ).set_output(transform="pandas")

    pipeline = Pipeline(steps=[
        ("preprocessing", preprocessor),
        ("elasticnet", ElasticNet(random_state=42, max_iter=20000))
    ])

    return pipeline

In [20]:
SEED = 42

In [21]:
DATA_PATH = "../data_cleaned/final_dataset.parquet"

In [ ]:
def run_optuna_search(X_global, X, y, gkf, groups, categorical_cols, all_boolean_cols, numerical_cols, n_trials=40):
    def objective(trial):
        alpha = trial.suggest_float("alpha", 1e-4, 10.0, log=True)
        l1_ratio = trial.suggest_float("l1_ratio", 0.05, 0.95)

        pipeline = create_pipeline(X_global, categorical_cols, all_boolean_cols, numerical_cols)

        pipeline.set_params(
            elasticnet__alpha=alpha,
            elasticnet__l1_ratio=l1_ratio
        )

        neg_mses = cross_val_score(
            pipeline, X, y,
            cv=gkf,                     # stratified cross validiation
            groups = groups,
            scoring="neg_mean_squared_error", # MSE
            n_jobs=-1,                  # parallelize
        )
        return (-1) * neg_mses.mean()

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=SEED),
        study_name="depression_elastic_net_tuning",
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study

In [23]:
df = pd.read_parquet(DATA_PATH)

df["any_disorder"] = df["any_disorder"].astype('category')

demographic_cols = [
    'age',
    'gender',
    'marital_status',
    'children',
    'ethnicity_1',
    'drop_school_1',
    'accommodation',
    'mh_smoking',
    'audit_1',
    'depression_medication'
]

categorical_cols = [
    'marital_status',
    'ethnicity_1',
    'accommodation',
    'mh_smoking',
    'audit_1',
]

boolean_cols_demographics = [
    'children',
    'drop_school_1',
    'depression_medication',
    'gender'
]

boolean_cols_sleep = [
    'any_disorder'
]

all_boolean_cols = boolean_cols_demographics + boolean_cols_sleep

numerical_cols = ['age']

group_cols = ['participant_name', 'timeInterval_ID']

outcome_col = ['phq8_score']
    

cols_to_exclude_numerical = demographic_cols + boolean_cols_sleep + group_cols + outcome_col

numerical_cols += [
    col for col in df.columns
    if col not in cols_to_exclude_numerical
]

X, y, groups = prepare_data(df, categorical_cols, boolean_cols_demographics)

In [24]:
outer_cv = GroupKFold(n_splits=5)
inner_cv = GroupKFold(n_splits=3)

test_rmse_scores = []
test_r2_scores = []
fold_params = []

In [25]:
for fold_idx, (train_idx, test_idx) in enumerate(
    outer_cv.split(X, y, groups), 1
):

    print(f"\nStarting outer fold {fold_idx}")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    groups_train = groups.iloc[train_idx]

    study = run_optuna_search(X, X_train, y_train, inner_cv, groups_train, categorical_cols, all_boolean_cols, numerical_cols)

    best_params = study.best_params

    print(f"Best params: {best_params}")

    fold_params.append(best_params)

    best_model = create_pipeline(X, categorical_cols, all_boolean_cols, numerical_cols)

    best_model.set_params(
        elasticnet__alpha=best_params["alpha"],
        elasticnet__l1_ratio=best_params["l1_ratio"]
    )

    best_model.fit(X_train, y_train)

    y_pred = best_model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    test_rmse_scores.append(rmse)
    test_r2_scores.append(r2)


Starting outer fold 1


  0%|          | 0/40 [00:00<?, ?it/s]

Best trial: 1. Best value: 34.7715:   8%|▊         | 3/40 [00:08<02:01,  3.28s/it]/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.037e+03, tolerance: 2.675e+01
  model = cd_fast.enet_coordinate_descent(
/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.386e+04, tolerance: 2.582e+01
  model = cd_fast.enet_coordinate_descent(
/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarnin

Best params: {'alpha': 0.10129197956845731, 'l1_ratio': 0.6872653200164409}

Starting outer fold 2


Best trial: 0. Best value: 34.3101:   8%|▊         | 3/40 [00:06<01:38,  2.65s/it]/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.642e+03, tolerance: 2.616e+01
  model = cd_fast.enet_coordinate_descent(
/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.701e+03, tolerance: 2.741e+01
  model = cd_fast.enet_coordinate_descent(
/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarnin

Best params: {'alpha': 0.11385821122717614, 'l1_ratio': 0.20686126545500846}

Starting outer fold 3


... (2 outputs left out)

Best trial: 1. Best value: 34.9382:   8%|▊         | 3/40 [00:05<01:26,  2.33s/it]/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.218e+03, tolerance: 2.679e+01
  model = cd_fast.enet_coordinate_descent(
/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.619e+03, tolerance: 2.719e+01
  model = cd_fast.enet_coordinate_descent(
/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarnin

Best params: {'alpha': 0.10129197956845731, 'l1_ratio': 0.6872653200164409}

Starting outer fold 5


Best trial: 0. Best value: 31.937:   8%|▊         | 3/40 [00:06<01:43,  2.79s/it]/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.270e+03, tolerance: 2.420e+01
  model = cd_fast.enet_coordinate_descent(
/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.828e+03, tolerance: 2.596e+01
  model = cd_fast.enet_coordinate_descent(
/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning

Best params: {'alpha': 0.05467530386354745, 'l1_ratio': 0.061859059217006296}


In [26]:
def print_cv_results(fold_params, test_rmse_scores, test_r2_scores):
    print("\n" + "="*50)
    print("CROSS-VALIDATION RESULTS")
    print("="*50)

    print("\nBest parameters in each fold:")
    for i, params in enumerate(fold_params, 1):
        print(f"  Fold {i}: {params}")

    print("\nTest metrics per fold:")
    for i, (rmse, r2) in enumerate(zip(test_rmse_scores, test_r2_scores), 1):
        print(f"  Fold {i}: RMSE = {rmse:.4f}, R² = {r2:.4f}")

    print("\nSummary:")
    print(f"  RMSE: {np.mean(test_rmse_scores):.4f} ± {np.std(test_rmse_scores):.4f}")
    print(f"  R²:   {np.mean(test_r2_scores):.4f} ± {np.std(test_r2_scores):.4f}")

    print("="*50 + "\n")

In [27]:
print_cv_results(
    fold_params,
    test_rmse_scores,
    test_r2_scores
)


CROSS-VALIDATION RESULTS

Best parameters in each fold:
  Fold 1: {'alpha': 0.10129197956845731, 'l1_ratio': 0.6872653200164409}
  Fold 2: {'alpha': 0.11385821122717614, 'l1_ratio': 0.20686126545500846}
  Fold 3: {'alpha': 0.11778986533056508, 'l1_ratio': 0.7098785099640688}
  Fold 4: {'alpha': 0.10129197956845731, 'l1_ratio': 0.6872653200164409}
  Fold 5: {'alpha': 0.05467530386354745, 'l1_ratio': 0.061859059217006296}

Test metrics per fold:
  Fold 1: RMSE = 5.7567, R² = 0.1392
  Fold 2: RMSE = 5.4127, R² = 0.1462
  Fold 3: RMSE = 5.7584, R² = 0.1182
  Fold 4: RMSE = 5.5718, R² = 0.1167
  Fold 5: RMSE = 6.0808, R² = 0.1505

Summary:
  RMSE: 5.7161 ± 0.2233
  R²:   0.1342 ± 0.0141

